# Task 023: lenstronomy_shapelets — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Galaxy light profile reconstruction using Shapelet basis decomposition

**Paper**: lenstronomy II: A gravitational lensing software ecosystem
**Repository**: https://github.com/lenstronomy/lenstronomy

### Physical Background

Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.

### Forward Model

```
y = f(theta) + n
```

### Inverse Problem

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

### Performance Metrics
- **PSNR**: 27.67 dB
- **SSIM**: N/A


### Mathematical Formulation

Gravitational lensing bends light according to the lens equation:

$$\boldsymbol{\beta} = \boldsymbol{\theta} - \boldsymbol{\alpha}(\boldsymbol{\theta})$$

where $\boldsymbol{\beta}$ is the source position, $\boldsymbol{\theta}$ is the image position, and $\boldsymbol{\alpha}$ is the deflection angle.

**Convergence**: $\kappa(\boldsymbol{\theta}) = \frac{\Sigma(\boldsymbol{\theta})}{\Sigma_{\text{cr}}}$

**SIE lens model** deflection:
$$\alpha_1 = \frac{\theta_E}{\sqrt{1-q^2}} \arctan\left(\frac{\sqrt{1-q^2}\,\theta_1}{\psi + q^2 s}\right)$$

**Source reconstruction**: $\hat{s} = \arg\min_s \|I_{\text{obs}} - L(\boldsymbol{\theta}_{\text{lens}}) s\|^2 + \lambda \|\nabla s\|_1$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import time
from lenstronomy.Util import image_util
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.ImSim.image_linear_solve import ImageLinearFit
from lenstronomy.LightModel.Profiles.shapelets import ShapeletSet

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
def load_and_preprocess_data(
    numPix: int,
    deltaPix: float,
    background_rms: float,
    exp_time: float,
    fwhm: float,
    kwargs_lens: list,
    kwargs_source_true: list,
    lens_model_list: list,
    source_model_list_true: list,
    random_seed: int = 42
) -> dict:
    """
    Load and preprocess data by simulating a lensed complex source with noise.
    
    Returns a dictionary containing all necessary data and class instances.
    """
    np.random.seed(random_seed)
    
    # Transformation matrix: pixel coordinates -> angular coordinates
    transform_pix2angle = np.array([[-deltaPix, 0], [0, deltaPix]])
    
    # Calculate the RA/Dec of the pixel (0,0) such that the image is centered at (0,0)
    cx = (numPix - 1) / 2.0
    cy = (numPix - 1) / 2.0
    ra_at_xy_0 = -(transform_pix2angle[0, 0] * cx + transform_pix2angle[0, 1] * cy)
    dec_at_xy_0 = -(transform_pix2angle[1, 0] * cx + transform_pix2angle[1, 1] * cy)
    
    # Initialize data class with zeros
    kwargs_data = {
        'background_rms': background_rms,
        'exposure_time': exp_time,
        'ra_at_xy_0': ra_at_xy_0,
        'dec_at_xy_0': dec_at_xy_0,
        'transform_pix2angle': transform_pix2angle,
        'image_data': np.zeros((numPix, numPix))
    }
    data_class = ImageData(**kwargs_data)
    
    # Configure PSF
    kwargs_psf = {
        'psf_type': 'GAUSSIAN',
        'fwhm': fwhm,
        'pixel_size': deltaPix,
        'truncation': 3
    }
    psf_class = PSF(**kwargs_psf)
    
    # Define Lens Model
    lens_model_class = LensModel(lens_model_list=lens_model_list)
    
    # Define True Source Model
    source_model_class_true = LightModel(light_model_list=source_model_list_true)
    
    # No Lens Light
    lens_light_model_list = []
    lens_light_model_class = LightModel(light_model_list=lens_light_model_list)
    
    # Numerics settings
    kwargs_numerics = {'supersampling_factor': 1, 'supersampling_convolution': False}
    
    # Create ImageModel for simulation
    imageModel = ImageModel(
        data_class, psf_class, lens_model_class, source_model_class_true,
        lens_light_model_class, kwargs_numerics=kwargs_numerics
    )
    
    # Simulate clean image
    image_sim = imageModel.image(kwargs_lens, kwargs_source_true, kwargs_lens_light=None, kwargs_ps=None)
    
    # Add Poisson Noise
    image_sim_counts = image_sim * exp_time
    image_sim_counts[image_sim_counts < 0] = 0
    poisson_counts = np.random.poisson(image_sim_counts)
    poisson = poisson_counts / exp_time
    poisson_noise = poisson - image_sim
    
    # Add Gaussian Background Noise
    bkg_noise = np.random.randn(*image_sim.shape) * background_rms
    
    # Combine to get noisy image
    image_noisy = image_sim + bkg_noise + poisson_noise
    
    # Update Data with Simulated Image
    kwargs_data['image_data'] = image_noisy
    data_class.update_data(image_noisy)
    
    return {
        'data_class': data_class,
        'psf_class': psf_class,
        'lens_model_class': lens_model_class,
        'kwargs_numerics': kwargs_numerics,
        'kwargs_lens': kwargs_lens,
        'image_data': image_noisy,
        'image_clean': image_sim,
        'numPix': numPix,
        'deltaPix': deltaPix,
        'background_rms': background_rms,
        'exp_time': exp_time
    }

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = f(theta) + n
```

Functions: `forward_operator`

In [ ]:
def forward_operator(
    shapelet_coeffs: np.ndarray,
    data_class: ImageData,
    psf_class: PSF,
    lens_model_class: LensModel,
    kwargs_lens: list,
    kwargs_source_template: list,
    kwargs_numerics: dict
) -> np.ndarray:
    """
    Forward operator: Given shapelet coefficients, compute the predicted lensed image.
    
    This implements the forward model: source -> lensing -> convolution with PSF -> image.
    """
    # Create source model for shapelets
    source_model_list = ['SHAPELETS']
    source_model_class = LightModel(light_model_list=source_model_list)
    
    # No lens light
    lens_light_model_class = LightModel(light_model_list=[])
    
    # Create ImageModel
    imageModel = ImageModel(
        data_class, psf_class, lens_model_class, source_model_class,
        lens_light_model_class, kwargs_numerics=kwargs_numerics
    )
    
    # Build kwargs_source with amplitudes
    n_max = kwargs_source_template[0]['n_max']
    beta = kwargs_source_template[0]['beta']
    center_x = kwargs_source_template[0]['center_x']
    center_y = kwargs_source_template[0]['center_y']
    
    # The ShapeletSet uses 'amp' as a list of coefficients
    kwargs_source = [{
        'n_max': n_max,
        'beta': beta,
        'center_x': center_x,
        'center_y': center_y,
        'amp': shapelet_coeffs
    }]
    
    # Compute predicted image
    y_pred = imageModel.image(kwargs_lens, kwargs_source, kwargs_lens_light=None, kwargs_ps=None)
    
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

Functions: `run_inversion`

In [ ]:
def run_inversion(
    data_class: ImageData,
    psf_class: PSF,
    lens_model_class: LensModel,
    kwargs_lens: list,
    kwargs_numerics: dict,
    n_max: int,
    beta: float,
    center_x: float = 0.0,
    center_y: float = 0.0
) -> dict:
    """
    Run the linear inversion to reconstruct the source using shapelets.
    
    Uses lenstronomy's ImageLinearFit to solve for shapelet coefficients.
    """
    # Define Reconstruction Model (Shapelets)
    source_model_list_reconstruct = ['SHAPELETS']
    source_model_class_reconstruct = LightModel(light_model_list=source_model_list_reconstruct)
    
    # Initialize ImageLinearFit
    imageLinearFit = ImageLinearFit(
        data_class=data_class,
        psf_class=psf_class,
        lens_model_class=lens_model_class,
        source_model_class=source_model_class_reconstruct,
        kwargs_numerics=kwargs_numerics
    )
    
    # Constraints for Shapelets (center and scale)
    kwargs_source_reconstruct = [{
        'n_max': n_max,
        'beta': beta,
        'center_x': center_x,
        'center_y': center_y
    }]
    
    start_time = time.time()
    
    # image_linear_solve returns: model_image, error_map, cov_param, param
    wls_model, error_map, cov_param, param = imageLinearFit.image_linear_solve(
        kwargs_lens=kwargs_lens,
        kwargs_source=kwargs_source_reconstruct,
        kwargs_lens_light=None,
        kwargs_ps=None,
        inv_bool=False
    )
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    # Calculate Reduced Chi-Square
    chi2_reduced = imageLinearFit.reduced_chi2(wls_model, error_map)
    
    # Extract Shapelet Coefficients
    shapelet_coeffs = np.array(param)
    
    return {
        'model_image': wls_model,
        'error_map': error_map,
        'cov_param': cov_param,
        'shapelet_coeffs': shapelet_coeffs,
        'chi2_reduced': chi2_reduced,
        'elapsed_time': elapsed_time,
        'kwargs_source_reconstruct': kwargs_source_reconstruct,
        'n_max': n_max,
        'beta': beta
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(
    inversion_result: dict,
    data_dict: dict
) -> dict:
    """
    Evaluate the results of the inversion.
    
    Computes metrics and returns summary statistics.
    """
    model_image = inversion_result['model_image']
    shapelet_coeffs = inversion_result['shapelet_coeffs']
    chi2_reduced = inversion_result['chi2_reduced']
    elapsed_time = inversion_result['elapsed_time']
    n_max = inversion_result['n_max']
    beta = inversion_result['beta']
    
    image_data = data_dict['image_data']
    background_rms = data_dict['background_rms']
    
    # Compute residuals
    residuals = image_data - model_image
    
    # Compute RMS of residuals
    residual_rms = np.sqrt(np.mean(residuals**2))
    
    # Compute peak signal-to-noise ratio
    signal_max = np.max(np.abs(model_image))
    psnr = 20 * np.log10(signal_max / residual_rms) if residual_rms > 0 else np.inf
    
    # Number of shapelet coefficients
    num_coeffs = len(shapelet_coeffs)
    
    # Expected number of coefficients for given n_max
    expected_num_coeffs = int((n_max + 1) * (n_max + 2) / 2)
    
    # Compute coefficient statistics
    coeff_mean = np.mean(shapelet_coeffs)
    coeff_std = np.std(shapelet_coeffs)
    coeff_max = np.max(np.abs(shapelet_coeffs))
    
    # Print evaluation results
    print(f"Reconstruction completed in {elapsed_time:.4f} seconds.")
    print(f"Reduced Chi^2: {chi2_reduced:.4f}")
    print(f"Number of Shapelet coefficients: {num_coeffs}")
    print(f"Expected coefficients for n_max={n_max}: {expected_num_coeffs}")
    print(f"Residual RMS: {residual_rms:.6f}")
    print(f"Background RMS: {background_rms:.6f}")
    print(f"PSNR: {psnr:.2f} dB")
    print(f"Shapelet beta (scale): {beta}")
    print(f"Coefficient mean: {coeff_mean:.6f}")
    print(f"Coefficient std: {coeff_std:.6f}")
    print(f"Coefficient max abs: {coeff_max:.6f}")
    
    return {
        'residuals': residuals,
        'residual_rms': residual_rms,
        'psnr': psnr,
        'chi2_reduced': chi2_reduced,
        'num_coeffs': num_coeffs,
        'elapsed_time': elapsed_time,
        'coeff_mean': coeff_mean,
        'coeff_std': coeff_std,
        'coeff_max': coeff_max
    }

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Define parameters
background_rms = 0.05
exp_time = 100
numPix = 100
deltaPix = 0.05
fwhm = 0.1

# Lens model parameters
lens_model_list = ['SIE', 'SHEAR']
kwargs_sie = {'theta_E': 1.0, 'e1': 0.1, 'e2': -0.1, 'center_x': 0, 'center_y': 0}
kwargs_shear = {'gamma1': 0.05, 'gamma2': 0.01}
kwargs_lens = [kwargs_sie, kwargs_shear]

# True source model parameters
source_model_list_true = ['SERSIC_ELLIPSE', 'SERSIC']
kwargs_source_true = [
    {'amp': 200, 'R_sersic': 0.3, 'n_sersic': 1, 'e1': 0.1, 'e2': 0.1, 'center_x': 0.1, 'center_y': 0.1},
    {'amp': 100, 'R_sersic': 0.1, 'n_sersic': 2, 'center_x': -0.2, 'center_y': 0.0}
]

# Shapelet reconstruction parameters
n_max = 8
beta = 0.2

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
print("Data generation complete (Lensed Complex Source).")
data_dict = load_and_preprocess_data(
    numPix=numPix,
    deltaPix=deltaPix,
    background_rms=background_rms,
    exp_time=exp_time,
    fwhm=fwhm,
    kwargs_lens=kwargs_lens,
    kwargs_source_true=kwargs_source_true,
    lens_model_list=lens_model_list,
    source_model_list_true=source_model_list_true,
    random_seed=42
)

### Step 2: Run inversion

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 2: Run inversion
print("Starting linear inversion for source reconstruction...")
inversion_result = run_inversion(
    data_class=data_dict['data_class'],
    psf_class=data_dict['psf_class'],
    lens_model_class=data_dict['lens_model_class'],
    kwargs_lens=data_dict['kwargs_lens'],
    kwargs_numerics=data_dict['kwargs_numerics'],
    n_max=n_max,
    beta=beta,
    center_x=0.0,
    center_y=0.0
)

### Step 3: Evaluate results

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 3: Evaluate results
evaluation = evaluate_results(inversion_result, data_dict)

### Step 4: Demonstrate forward operator

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 4: Demonstrate forward operator
# Verify forward operator produces same result as inversion model
y_pred = forward_operator(
    shapelet_coeffs=inversion_result['shapelet_coeffs'],
    data_class=data_dict['data_class'],
    psf_class=data_dict['psf_class'],
    lens_model_class=data_dict['lens_model_class'],
    kwargs_lens=data_dict['kwargs_lens'],
    kwargs_source_template=inversion_result['kwargs_source_reconstruct'],
    kwargs_numerics=data_dict['kwargs_numerics']
)

forward_diff = np.max(np.abs(y_pred - inversion_result['model_image']))
print(f"Forward operator consistency check (max diff): {forward_diff:.2e}")

print("Reconstruction successful.")
print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **lenstronomy_shapelets**:

1. **Problem Setup**: Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=27.67 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: lenstronomy II: A gravitational lensing software ecosystem
- Repository: https://github.com/lenstronomy/lenstronomy
